# CS610 Integrated Risk Scoring Pipeline
**Pillar 1** (40%) — TF-IDF Identity Resolution  
**Pillar 2** (30%) — Crime Severity (crime categorisation export)  
**Pillar 3** (confirmation gate) — Biometric Prediction (Ensemble Voting)  
**Pillar 4** (20%) — Hidden Linkage (GCN/GraphSAGE)  
**Pillar 5** (10%) — Visual Similarity (CLIP)  

---

### Workflow
1. Client submits a query (name + biometrics)
2. Candidate retrieval: TF-IDF finds closest fugitive match
3. **P3**: Biometric ensemble compares client vs matched fugitive → `p3_confidence`
4. **If P3 MATCH** (biometrics confirm identity) → immediate **CRITICAL** escalation — no further scoring needed
5. **If P3 MISMATCH** (biometrics inconclusive/conflict) → proceed with full multi-pillar evaluation:
   - **P1**: Identity resolution score → `identity_score`
   - **P2**: Crime severity → `crime_severity`
   - **P4**: Hidden linkage → `linkage_score`
   - **P5**: Visual similarity → `visual_score`
   - **Final Risk** = P1×0.40 + P2×0.30 + P4×0.20 + P5×0.10

### Rationale
If biometrics confirm the match, we have direct physical evidence — the strongest signal. If biometrics *don't* confirm, that does not clear the client; the name may still match, the person may have altered their appearance, or biometric data may be incomplete. In that case, we need *all* pillars to triangulate the risk.

### P4 Formula (normalised to [0, 1])
Rank weights (0.50 / 0.30 / 0.20) are normalised by the sum of available ranks so P4 ∈ [0, 1] even when fewer than 3 links exist.

`P4 = Sum (rank_weight_k / weight_sum) × link_score_k × severity_k`

## Import Libraries

In [17]:
import os
import pickle
import warnings
from datetime import datetime
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Load Data & Pre-Computed Assets

In [18]:
# ═══════════════════════════════════════════════════════════════════════════════
# DIRECTORY CONFIGURATION — each pillar's own output directory
# ═══════════════════════════════════════════════════════════════════════════════

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))   # CS610_Interpol/


def _resolve(*candidates):
    """Return the first path that exists, or the last candidate as default."""
    for p in candidates:
        if os.path.exists(p):
            return p
    return candidates[-1]


# ── Pillar base directories ──────────────────────────────────────────────────
P1_BASE = os.path.join(REPO_ROOT, 'Pillar 1 (Identity Resolution)')
P2_BASE = os.path.join(REPO_ROOT, 'Pillar 2 (Crime Severity)')
P3_BASE = os.path.join(REPO_ROOT, 'Pillar 3 (Biometric Prediction)')
P4_BASE = os.path.join(REPO_ROOT, 'Pillar 4 (GCN and Graphsage)')
P5_BASE = os.path.join(REPO_ROOT, 'Pillar 5 (Visual)')

P4_OUT_DIR = os.path.join(P4_BASE, 'outputs')

# ── File paths (resolved from each pillar's actual output location) ──────────
FUGITIVE_CSV = os.path.join(P4_BASE, 'crime_analysis_results_aft_transformer_ner.csv')

# P1: Pillar 1 saves vectorizer+embeddings into its own outputs/ directory
P1_VECTORIZER_PKL = _resolve(
    os.path.join(P1_BASE, 'outputs', 'pillar1_name_vectorizer.pkl'),
    os.path.join(P1_BASE, 'output',  'pillar1_name_vectorizer.pkl'),
    os.path.join(REPO_ROOT, 'outputs', 'pillar1_name_vectorizer.pkl'),   # legacy
)
P1_EMBEDDINGS_PKL = _resolve(
    os.path.join(P1_BASE, 'outputs', 'pillar1_name_embeddings.pkl'),
    os.path.join(P1_BASE, 'output',  'pillar1_name_embeddings.pkl'),
    os.path.join(REPO_ROOT, 'outputs', 'pillar1_name_embeddings.pkl'),   # legacy
)

# P2: Pillar 2 exports CSV in its pillar root directory (no outputs/ subfolder)
P2_CRIME_CSV = _resolve(
    os.path.join(P2_BASE, 'crime_categorisation_export.csv'),
    os.path.join(P2_BASE, 'outputs', 'crime_categorisation_export.csv'),
)

# P3: Pillar 3 notebook saves pickles to ../outputs/ (repo-root outputs)
P3_MODEL_PKL = _resolve(
    os.path.join(REPO_ROOT, 'outputs', 'pillar3_ensemble_voting.pkl'),
    os.path.join(P3_BASE, 'output', 'ensemble_voting.pkl'),             # legacy
    os.path.join(P3_BASE, 'outputs', 'ensemble_voting.pkl'),
)

CLIENT_CSV    = os.path.join(REPO_ROOT, 'synthetic_data_generation', 'synthetic_client_data.csv')
P4_LINKS_CSV  = os.path.join(P4_OUT_DIR, 'top_new_links_all.csv')
P4_SCORES_CSV = os.path.join(P4_OUT_DIR, 'hidden_linkage_scores.csv')

# P5: not ready yet
P5_DIR        = os.path.join(P5_BASE, 'outputs')
P5_VISUAL_CSV = os.path.join(P5_DIR, 'visual_scores.csv')

# ── Integration output ────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Verify paths ──────────────────────────────────────────────────────────────
print(f'Repo root  : {REPO_ROOT}')
print(f'P1 base    : {P1_BASE}')
print(f'P2 base    : {P2_BASE}')
print(f'P3 base    : {P3_BASE}')
print(f'P4 base    : {P4_BASE}')
print(f'P5 base    : {P5_BASE}')
print(f'Output dir : {OUTPUT_DIR}')
print()
for label, path in [('P1 vectorizer', P1_VECTORIZER_PKL), ('P1 embeddings', P1_EMBEDDINGS_PKL),
                     ('P2 crime CSV', P2_CRIME_CSV), ('P3 model', P3_MODEL_PKL),
                     ('P4 links', P4_LINKS_CSV), ('P4 scores', P4_SCORES_CSV),
                     ('Fugitive CSV', FUGITIVE_CSV), ('Client CSV', CLIENT_CSV)]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'  {status} {label:15s}: {path}')
print()

# ═══════════════════════════════════════════════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════════════════════════════════════════════

# ── Pillar 2: Crime severity from crime categorisation export ─────────────────
df_crime_cat = pd.read_csv(P2_CRIME_CSV)
print(f'✅ P2 Crime categorisation loaded — {len(df_crime_cat):,} rows')

crime_severity_map = df_crime_cat.set_index('id')[['final_crime_label', 'severity_score', 'risk_tier']].to_dict('index')
print(f'   Severity map: {len(crime_severity_map):,} fugitives')
print(f'   Crime labels: {sorted(df_crime_cat["final_crime_label"].unique())}')

# ── Pillar 3: Biometric ensemble model + client database ─────────────────────
if os.path.exists(P3_MODEL_PKL):
    with open(P3_MODEL_PKL, 'rb') as f:
        p3_model = pickle.load(f)
    print(f'✅ P3 Biometric ensemble model loaded — {P3_MODEL_PKL}')
else:
    p3_model = None
    print(f'⚠️  P3 Model not found — biometric validation will be skipped')

if os.path.exists(CLIENT_CSV):
    df_clients = pd.read_csv(CLIENT_CSV)
    df_clients['full_name_upper'] = df_clients['full_name'].str.upper().str.strip()
    df_clients = df_clients.drop_duplicates(subset='full_name_upper', keep='first')
    client_bio_map = df_clients.set_index('full_name_upper').to_dict('index')
    print(f'✅ Client biometric database loaded — {len(df_clients):,} clients')
else:
    df_clients = pd.DataFrame()
    client_bio_map = {}
    print(f'⚠️  Client CSV not found — P3 will be skipped')

# ── Main fugitive dataset ─────────────────────────────────────────────────────
df_fugitives = pd.read_csv(FUGITIVE_CSV)
df_unique = df_fugitives.drop_duplicates(subset='id', keep='first').reset_index(drop=True)
df_p1_lookup = df_fugitives.reset_index(drop=True)  # aligned with pickle embeddings

df_unique['final_crime_label'] = df_unique['id'].map(
    lambda x: crime_severity_map.get(x, {}).get('final_crime_label', 'Unknown')
)
df_unique['severity_score'] = df_unique['id'].map(
    lambda x: crime_severity_map.get(x, {}).get('severity_score', 0.1)
)
print(f'✅ Fugitive database loaded — {len(df_unique):,} unique fugitives')
print(f'   Enriched with crime categorisation: {(df_unique["final_crime_label"] != "Unknown").sum():,} matched')

# ── Pillar 1: TF-IDF vectorizer + embeddings ─────────────────────────────────
if os.path.exists(P1_VECTORIZER_PKL) and os.path.exists(P1_EMBEDDINGS_PKL):
    with open(P1_VECTORIZER_PKL, 'rb') as f:
        name_vectorizer = pickle.load(f)
    with open(P1_EMBEDDINGS_PKL, 'rb') as f:
        name_embeddings = pickle.load(f)
    if name_embeddings.shape[0] != len(df_p1_lookup):
        print(f'⚠️  Pickle embeddings ({name_embeddings.shape[0]}) != fugitive rows ({len(df_p1_lookup)}) — rebuilding on deduped set')
        df_p1_lookup = df_unique.reset_index(drop=True)
        name_embeddings = normalize(
            name_vectorizer.fit_transform(df_p1_lookup['name'].str.upper().astype(str))
        )
    print(f'✅ P1 Loaded from pickle')
    print(f'   Vectorizer : {P1_VECTORIZER_PKL}')
    print(f'   Embeddings : {P1_EMBEDDINGS_PKL}  shape={name_embeddings.shape}')
    print(f'   P1 lookup rows: {len(df_p1_lookup)}')
else:
    name_vectorizer = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(2, 4),
        max_features=5000, sublinear_tf=True
    )
    df_p1_lookup = df_unique.reset_index(drop=True)
    name_embeddings = normalize(
        name_vectorizer.fit_transform(df_p1_lookup['name'].str.upper().astype(str))
    )
    print(f'⚠️  P1 Pickle not found — rebuilt TF-IDF from scratch')
    print(f'   Embeddings shape: {name_embeddings.shape}')

# ── Pillar 4: Pre-computed top-3 links from GraphSAGE ────────────────────────
if os.path.exists(P4_LINKS_CSV):
    df_links = pd.read_csv(P4_LINKS_CSV)
    print(f'✅ P4 Link predictions loaded — {len(df_links)} rows, '
          f'{df_links["anchor_id"].nunique()} anchors')
else:
    df_links = pd.DataFrame()
    print(f'⚠️  P4 top_new_links_all.csv not found — linkage scores will be 0')

# ── Pillar 4: Pre-computed mean linkage scores ───────────────────────────────
if os.path.exists(P4_SCORES_CSV):
    df_linkage_scores = pd.read_csv(P4_SCORES_CSV)
    linkage_score_map = df_linkage_scores.set_index('id')['linkage_score'].to_dict()
    print(f'✅ P4 Hidden linkage scores loaded — {len(df_linkage_scores):,} fugitives')
else:
    df_linkage_scores = pd.DataFrame()
    linkage_score_map = {}
    print(f'⚠️  P4 hidden_linkage_scores.csv not found')

# ── Pillar 5: Visual similarity (not ready) ──────────────────────────────────
if os.path.exists(P5_VISUAL_CSV):
    df_visual = pd.read_csv(P5_VISUAL_CSV)
    print(f'✅ P5 Visual scores loaded — {len(df_visual)} rows')
else:
    df_visual = None
    print('⚠️  P5 not ready — using default score 0.5')

Repo root  : /Volumes/My Mini Drive/CS 610/CS610_Interpol
P1 base    : /Volumes/My Mini Drive/CS 610/CS610_Interpol/Pillar 1 (Identity Resolution)
P2 base    : /Volumes/My Mini Drive/CS 610/CS610_Interpol/Pillar 2 (Crime Severity)
P3 base    : /Volumes/My Mini Drive/CS 610/CS610_Interpol/Pillar 3 (Biometric Prediction)
P4 base    : /Volumes/My Mini Drive/CS 610/CS610_Interpol/Pillar 4 (GCN and Graphsage)
P5 base    : /Volumes/My Mini Drive/CS 610/CS610_Interpol/Pillar 5 (Visual)
Output dir : /Volumes/My Mini Drive/CS 610/CS610_Interpol/Integration/outputs

  ✅ P1 vectorizer  : /Volumes/My Mini Drive/CS 610/CS610_Interpol/Pillar 1 (Identity Resolution)/output/pillar1_name_vectorizer.pkl
  ✅ P1 embeddings  : /Volumes/My Mini Drive/CS 610/CS610_Interpol/Pillar 1 (Identity Resolution)/output/pillar1_name_embeddings.pkl
  ✅ P2 crime CSV   : /Volumes/My Mini Drive/CS 610/CS610_Interpol/Pillar 2 (Crime Severity)/crime_categorisation_export.csv
  ✅ P3 model       : /Volumes/My Mini Drive/CS 61

## 2. Define Constants

In [19]:
# ── Crime severity weights (from crime_categorisation_export.csv) ────────────
# Build mapping from crime label → severity score using the teammate's export
CRIME_SEVERITY = df_crime_cat.groupby('final_crime_label')['severity_score'].first().to_dict()
CRIME_SEVERITY['Unknown'] = 0.1  # Fallback for unmatched fugitives
print(f'Crime severity mapping (from CSV):')
for label, score in sorted(CRIME_SEVERITY.items(), key=lambda x: -x[1]):
    print(f'  {label:<20} → {score}')

# ── Pillar weights ────────────────────────────────────────────────────────────
W_P1 = 0.40   # Identity Resolution (TF-IDF cosine similarity)
W_P2 = 0.30   # Crime Severity
W_P4 = 0.20   # Hidden Linkage (GraphSAGE)
W_P5 = 0.10   # Visual Similarity (CLIP)

# ── Pillar 4 internal rank weights ────────────────────────────────────────────
LINK_RANK_WEIGHTS = {1: 0.50, 2: 0.30, 3: 0.20}

# ── Final risk thresholds (AML industry norms) ────────────────────────────────
FINAL_THRESHOLDS = {
    'CRITICAL':  0.70,
    'HIGH RISK': 0.50,
    'REVIEW':    0.30,
    'LOW RISK':  0.00,
}

# ── Default visual score when P5 is not available ─────────────────────────────
DEFAULT_VISUAL_SCORE = 0.5

print(f'\nConstants loaded.')
print(f'  Pillar weights: P1={W_P1}, P2={W_P2}, P4={W_P4}, P5={W_P5} (sum={W_P1+W_P2+W_P4+W_P5})')
print(f'  P4 rank weights: {LINK_RANK_WEIGHTS} (sum={sum(LINK_RANK_WEIGHTS.values())})')

Crime severity mapping (from CSV):
  terrorism            → 1.0
  homicide             → 0.9
  sexual crime         → 0.8
  armed formation      → 0.7
  assault              → 0.7
  narcotics            → 0.7
  financial crime      → 0.5
  Unknown              → 0.1

Constants loaded.
  Pillar weights: P1=0.4, P2=0.3, P4=0.2, P5=0.1 (sum=0.9999999999999999)
  P4 rank weights: {1: 0.5, 2: 0.3, 3: 0.2} (sum=1.0)


## 3. Pipeline Functions

In [20]:
# ═════════════════════════════════════════════════════════════════════════════
# PILLAR 1 — Identity Resolution
# ═════════════════════════════════════════════════════════════════════════════

def pillar1_identity(client_name: str) -> dict:
    """
    TF-IDF char n-gram cosine similarity against the fugitive database.
    Returns the top match with score and index.
    """
    q = normalize(name_vectorizer.transform([client_name.upper()]))
    sims = cosine_similarity(q, name_embeddings)[0]
    top_idx = int(np.argmax(sims))
    top_score = float(sims[top_idx])
    fugitive = df_p1_lookup.iloc[top_idx]
    
    # Top-3 for reporting
    top3_idx = np.argsort(sims)[::-1][:3]
    top3 = [(df_p1_lookup.iloc[i]['name'], round(float(sims[i]), 4)) for i in top3_idx]
    
    return {
        'identity_score': top_score,
        'matched_id': fugitive['id'],
        'matched_name': fugitive['name'],
        'matched_crime': fugitive.get('final_crime_label', 'Unknown'),
        'top3': top3,
        'matched_idx': top_idx,
    }

In [21]:
# ═════════════════════════════════════════════════════════════════════════════
# PILLAR 2 — Crime Severity (from crime_categorisation_export.csv)
# ═════════════════════════════════════════════════════════════════════════════

def pillar2_crime_severity(fugitive_id: str, crime_type: str = None) -> tuple:
    """
    Look up crime severity from the crime categorisation export.
    Falls back to CRIME_SEVERITY dict lookup by crime_type.
    Returns (crime_label, severity_score).
    """
    # Primary: look up by fugitive ID in the crime categorisation export
    if fugitive_id in crime_severity_map:
        entry = crime_severity_map[fugitive_id]
        return entry['final_crime_label'], entry['severity_score']
    
    # Fallback: use the crime type from the original dataset
    if crime_type and crime_type in CRIME_SEVERITY:
        return crime_type, CRIME_SEVERITY[crime_type]
    
    return 'Unknown', CRIME_SEVERITY['Unknown']

In [22]:
# ═════════════════════════════════════════════════════════════════════════════
# PILLAR 3 — Biometric Prediction (Ensemble Voting)
# ═════════════════════════════════════════════════════════════════════════════

P3_THRESHOLD = 0.5

def _parse_dob(val):
    """Parse a date-of-birth string into a datetime, or return None."""
    if pd.isna(val) or str(val).strip() in ('', 'nan', 'NaN'):
        return None
    try:
        return pd.to_datetime(str(val).strip())
    except Exception:
        return None

def compute_p3_features(client: dict, fugitive) -> list:
    """
    Compute the 7 biometric pairwise features used by Pillar 3.
    Mirrors the feature engineering in Pillar 3 (Biometric Prediction).ipynb.
    Missing values are filled with 0 (consistent with training preprocessing).
    """
    client_name = str(client.get('full_name', ''))
    fug_name    = str(fugitive.get('name', '') if isinstance(fugitive, dict) else fugitive.get('name', ''))

    # 1. name_similarity
    name_sim = SequenceMatcher(None, client_name.upper(), fug_name.upper()).ratio()

    # 2. age_difference
    today = datetime.today()
    client_dob = _parse_dob(client.get('date_of_birth'))
    fug_dob    = _parse_dob(fugitive.get('birth_date') if isinstance(fugitive, dict) else fugitive.get('birth_date'))
    if client_dob and fug_dob:
        client_age = relativedelta(today, client_dob).years
        fug_age    = relativedelta(today, fug_dob).years
        age_diff   = abs(client_age - fug_age)
    else:
        age_diff = 0

    # 3. same_gender
    client_gender = str(client.get('gender', '')).strip().upper()
    fug_gender    = str(fugitive.get('GENDER', '') if isinstance(fugitive, dict) else fugitive.get('GENDER', '')).strip().upper()
    same_gender = 1 if (client_gender and fug_gender and client_gender == fug_gender) else 0

    # 4. height_difference
    try:
        c_h = float(client.get('height_m', 0) or 0)
    except (ValueError, TypeError):
        c_h = 0.0
    try:
        f_h = float(fugitive.get('height', 0) if isinstance(fugitive, dict) else fugitive.get('height', 0) or 0)
    except (ValueError, TypeError):
        f_h = 0.0
    height_diff = abs(c_h - f_h)

    # 5. weight_difference (fugitive DB lacks weight → defaults to 0)
    try:
        c_w = float(client.get('weight_kg', 0) or 0)
    except (ValueError, TypeError):
        c_w = 0.0
    try:
        f_w = float(fugitive.get('weight', 0) if isinstance(fugitive, dict) else fugitive.get('weight', 0) or 0)
    except (ValueError, TypeError):
        f_w = 0.0
    weight_diff = abs(c_w - f_w)

    # 6. same_hair_colour
    c_hair = str(client.get('hair_color', '')).strip().upper()
    f_hair = str(fugitive.get('hairColor', '') if isinstance(fugitive, dict) else fugitive.get('hairColor', '')).strip().upper()
    same_hair = 1 if (c_hair and f_hair and c_hair not in ('', 'NAN') and f_hair not in ('', 'NAN', 'OTHD') and c_hair == f_hair) else 0

    # 7. same_eye_colour
    c_eye = str(client.get('eye_color', '')).strip().upper()
    f_eye = str(fugitive.get('eyeColor', '') if isinstance(fugitive, dict) else fugitive.get('eyeColor', '')).strip().upper()
    same_eye = 1 if (c_eye and f_eye and c_eye not in ('', 'NAN') and f_eye not in ('', 'NAN', 'OTHD') and c_eye == f_eye) else 0

    return [name_sim, age_diff, same_gender, height_diff, weight_diff, same_hair, same_eye]


def pillar3_biometric(client_name: str, matched_fugitive) -> dict:
    """
    Run P3 ensemble model on a client–fugitive pair.
    Returns confidence score (probability of match) and binary match prediction.
    """
    if p3_model is None:
        return {'p3_confidence': None, 'p3_match': None, 'p3_note': 'Model not loaded'}

    client_key = client_name.upper().strip()
    client = client_bio_map.get(client_key, None)
    if client is None:
        return {'p3_confidence': None, 'p3_match': None, 'p3_note': 'Client not in biometric database'}

    features = compute_p3_features(client, matched_fugitive)
    features = [0.0 if (x is None or (isinstance(x, float) and np.isnan(x))) else x for x in features]
    proba = p3_model.predict_proba([features])[0]
    confidence = float(proba[1])

    return {
        'p3_confidence': round(confidence, 4),
        'p3_match': int(confidence >= P3_THRESHOLD),
        'p3_features': {
            'name_similarity': round(features[0], 4),
            'age_difference': features[1],
            'same_gender': features[2],
            'height_difference': round(features[3], 4),
            'weight_difference': round(features[4], 4),
            'same_hair_colour': features[5],
            'same_eye_colour': features[6],
        },
    }

print(f'P3 threshold: {P3_THRESHOLD}')
print(f'P3 feature columns: name_similarity, age_difference, same_gender, height_difference, weight_difference, same_hair_colour, same_eye_colour')

P3 threshold: 0.5
P3 feature columns: name_similarity, age_difference, same_gender, height_difference, weight_difference, same_hair_colour, same_eye_colour


In [23]:
# ═════════════════════════════════════════════════════════════════════════════
# PILLAR 4 — Hidden Linkage (GraphSAGE)
# ═════════════════════════════════════════════════════════════════════════════

def pillar4_linkage(matched_id: str) -> dict:
    """
    Look up top-3 linked fugitives from pre-computed GraphSAGE predictions.
    
    Formula (normalised to [0, 1]):
      weight_sum = sum of LINK_RANK_WEIGHTS for available ranks
      P4 = sum over available rank k:
           (LINK_RANK_WEIGHTS[k] / weight_sum) × link_score_k × crime_severity_k
    
    When all 3 links present → weights stay 0.50 / 0.30 / 0.20
    When 2 links present   → weights become 0.625 / 0.375
    When 1 link present    → weight becomes 1.0
    """
    links = df_links[df_links['anchor_id'] == matched_id].sort_values('rank')
    
    if links.empty:
        fallback = linkage_score_map.get(matched_id, 0.0)
        return {
            'linkage_score': round(fallback, 4),
            'linked_fugitives': [],
            'note': 'Score from hidden_linkage_scores.csv (no per-link detail available)' if fallback > 0 else 'No linkage data found',
        }
    
    # Normalise rank weights so they sum to 1.0 regardless of how many links exist
    available_ranks = [int(row['rank']) for _, row in links.iterrows()]
    weight_sum = sum(LINK_RANK_WEIGHTS.get(r, 0.0) for r in available_ranks)
    
    breakdown = []
    total = 0.0
    
    for _, row in links.iterrows():
        rank = int(row['rank'])
        link_score = float(row['predicted_link_score'])
        
        crime_label, severity = pillar2_crime_severity(
            row['candidate_id'],
            df_unique[df_unique['id'] == row['candidate_id']].iloc[0].get('detected_crime_type', 'Unknown')
            if not df_unique[df_unique['id'] == row['candidate_id']].empty else 'Unknown'
        )
        
        raw_weight = LINK_RANK_WEIGHTS.get(rank, 0.0)
        norm_weight = raw_weight / weight_sum if weight_sum > 0 else 0.0
        contribution = norm_weight * link_score * severity
        total += contribution
        
        breakdown.append({
            'rank': rank,
            'candidate_name': row['candidate_name'],
            'candidate_id': row['candidate_id'],
            'link_score': round(link_score, 4),
            'crime_type': crime_label,
            'crime_severity': severity,
            'rank_weight': round(norm_weight, 4),
            'contribution': round(contribution, 4),
        })
    
    return {
        'linkage_score': round(total, 4),
        'linked_fugitives': breakdown,
    }

In [24]:
# ═════════════════════════════════════════════════════════════════════════════
# PILLAR 5 — Visual Similarity (CLIP)
# ═════════════════════════════════════════════════════════════════════════════

def pillar5_visual(client_name: str, matched_id: str) -> float:
    """
    Look up pre-computed visual similarity score.
    Falls back to DEFAULT_VISUAL_SCORE if not available.
    """
    if df_visual is None:
        return DEFAULT_VISUAL_SCORE
    
    # Try to find a match by client_query and matched_fugitive_id
    match = df_visual[
        (df_visual['client_query'].str.upper() == client_name.upper()) &
        (df_visual['matched_fugitive_id'] == matched_id)
    ]
    
    if not match.empty:
        return float(match.iloc[0]['visual_score'])
    
    # Fallback: try matching by fugitive ID only
    match_by_id = df_visual[df_visual['matched_fugitive_id'] == matched_id]
    if not match_by_id.empty:
        return float(match_by_id.iloc[0]['visual_score'])
    
    return DEFAULT_VISUAL_SCORE

In [25]:
# ═════════════════════════════════════════════════════════════════════════════
# FINAL RISK SCORING
# ═════════════════════════════════════════════════════════════════════════════

def classify_risk(score: float) -> tuple:
    """
    Classify final risk score into tiers.
    Returns (tier, action, icon).
    """
    if score >= FINAL_THRESHOLDS['CRITICAL']:
        return ('CRITICAL',
                'Immediate escalation — freeze transaction, notify compliance officer',
                '🔴')
    elif score >= FINAL_THRESHOLDS['HIGH RISK']:
        return ('HIGH RISK',
                'Senior analyst review required within 24 hours',
                '🟠')
    elif score >= FINAL_THRESHOLDS['REVIEW']:
        return ('REVIEW',
                'Junior analyst flag — monitor and log for 30 days',
                '🟡')
    else:
        return ('LOW RISK',
                'Pass — logged to audit trail, no action required',
                '✅')


def screen_client(client_name: str) -> dict:
    """
    Full integrated screening pipeline.

    Workflow:
      1. Candidate retrieval via TF-IDF (find who to compare against)
      2. P3: Biometric comparison → confidence score
      3. If P3 MATCH (confirmed) → CRITICAL escalation (no further scoring)
         If P3 MISMATCH / N/A  → full P1 + P2 + P4 + P5 → Final Risk
    """
    # ── Candidate retrieval (TF-IDF) ──────────────────────────────────────────
    p1 = pillar1_identity(client_name)
    matched_fug = df_unique[df_unique['id'] == p1['matched_id']].iloc[0]

    # ── Pillar 3: Biometric comparison ────────────────────────────────────────
    p3 = pillar3_biometric(client_name, matched_fug)

    p3_confirmed = (p3['p3_match'] == 1) if p3['p3_match'] is not None else False

    if p3_confirmed:
        # Biometrics confirm identity → strongest evidence → immediate CRITICAL
        tier = 'CRITICAL'
        action = 'Biometric match confirmed — freeze + escalate immediately'
        icon = '🔴'
        p2_label, p2_score = pillar2_crime_severity(
            p1['matched_id'], p1['matched_crime']
        )
        return {
            'client_name': client_name,
            'matched_id': p1['matched_id'],
            'matched_name': p1['matched_name'],
            'matched_crime': p2_label,
            'top3_matches': p1['top3'],
            'p3_confidence': p3['p3_confidence'],
            'p3_match': p3['p3_match'],
            'p3_features': p3.get('p3_features'),
            'p3_note': p3.get('p3_note'),
            'p1_identity_score': round(p1['identity_score'], 4),
            'p2_crime_severity': p2_score,
            'p4_linkage_score': 0.0,
            'p4_breakdown': [],
            'p5_visual_score': 0.0,
            'final_risk': 1.0,
            'tier': tier,
            'action': action,
            'icon': icon,
        }

    # ── P3 mismatch / unavailable → need all pillars to evaluate ──────────────

    # ── Pillar 2 (from crime categorisation export) ───────────────────────────
    p2_label, p2_score = pillar2_crime_severity(
        p1['matched_id'], p1['matched_crime']
    )

    # ── Pillar 4 ──────────────────────────────────────────────────────────────
    p4 = pillar4_linkage(p1['matched_id'])

    # ── Pillar 5 ──────────────────────────────────────────────────────────────
    p5_score = pillar5_visual(client_name, p1['matched_id'])

    # ── Final Score ───────────────────────────────────────────────────────────
    final_risk = (
        p1['identity_score'] * W_P1 +
        p2_score             * W_P2 +
        p4['linkage_score']  * W_P4 +
        p5_score             * W_P5
    )

    tier, action, icon = classify_risk(final_risk)

    return {
        'client_name': client_name,
        'matched_id': p1['matched_id'],
        'matched_name': p1['matched_name'],
        'matched_crime': p2_label,
        'top3_matches': p1['top3'],
        'p3_confidence': p3['p3_confidence'],
        'p3_match': p3['p3_match'],
        'p3_features': p3.get('p3_features'),
        'p3_note': p3.get('p3_note'),
        'p1_identity_score': round(p1['identity_score'], 4),
        'p2_crime_severity': p2_score,
        'p4_linkage_score': p4['linkage_score'],
        'p4_breakdown': p4['linked_fugitives'],
        'p5_visual_score': round(p5_score, 4),
        'final_risk': round(final_risk, 4),
        'tier': tier,
        'action': action,
        'icon': icon,
    }

## 4. Run Test Cases (from Pillar 1)

In [26]:
# ═════════════════════════════════════════════════════════════════════════════
# TEST CASES — Aligned with Pillar 1 queries from CS610_Project.ipynb
# ═════════════════════════════════════════════════════════════════════════════

TEST_CASES = [
    # (client_query, description)
    ('S. Nikitenko',              'Initial + surname'),
    ('Hasen Aksema',              'Missing middle name'),
    ('Abdul Sambolotov',          'Single char typo'),
    ('JIAN XIA',                  'Exact match'),
    ('Norbert Bialas',            'Case mismatch'),
    ('Ramazan Chigayev',          'Transliteration variant'),
    ('M. Al-Saeed',               'Abbreviated first name'),
    ('Levitan Elyasov',           'Cross-script: и→i, я→ya'),
    
    # Real INTERPOL fugitive name variants
    ('Jose Solares Gonzalez',     'Missing middle name'),
    ('J. Antonio Solares',        'Initial + dropped surname'),
    ('Gonzalez Jose Antonio',     'Reversed name order'),
    ('Shyamal Rao',               'Missing surname'),
    ('S. Rao Reddy',              'Initial + surname'),
    ('Shyamal Roa Reddy',         'Typo in middle name'),
    ('M. James Winters',          'Initial + surname'),
    ('Marlon Winters',            'Missing middle name'),
    ('Marlon James Wynters',      'Typo in surname'),
]

In [27]:
# ═════════════════════════════════════════════════════════════════════════════
# RUN INTEGRATED SCREENING
# ═════════════════════════════════════════════════════════════════════════════

print('=' * 90)
print('  INTEGRATED RISK SCORING PIPELINE')
print('  Formula: P1(0.40) + P2(0.30) + P4(0.20) + P5(0.10)')
print('=' * 90)

all_results = []

for client_name, desc in TEST_CASES:
    result = screen_client(client_name)
    all_results.append(result)
    
    print(f'\n{result["icon"]} {result["tier"]} — Client: "{client_name}" ({desc})')
    print(f'  Matched Fugitive : {result["matched_name"]} [{result["matched_id"]}]')
    print(f'  Matched Crime    : {result["matched_crime"]}')
    p3c = result.get('p3_confidence')
    p3_str = f'{p3c:.4f}' if p3c is not None else 'N/A'
    p3_flag = '✓ MATCH' if result.get('p3_match') == 1 else ('✗ NO MATCH' if result.get('p3_match') == 0 else (result.get('p3_note', '')))
    print(f'  ┌─────────────────────────────────────────────────────────────')
    print(f'  │ P3 Biometric Conf. : {p3_str}  ({p3_flag})')
    if result.get('p3_match') == 1:
        # P3 confirmed → biometric match is the strongest evidence
        print(f'  ├─────────────────────────────────────────────────────────────')
        print(f'  │ 🔴 P3 biometric CONFIRMED — immediate CRITICAL escalation')
        print(f'  │ FINAL RISK SCORE   : {result["final_risk"]:.4f}')
        print(f'  └─────────────────────────────────────────────────────────────')
        print(f'  Action: {result["action"]}')
    else:
        # P3 mismatch or N/A → full multi-pillar evaluation
        if result.get('p3_match') == 0:
            print(f'  │ ↳ Mismatch — proceeding with full pillar evaluation')
        print(f'  │ P1 Identity Score  : {result["p1_identity_score"]:.4f}  × {W_P1} = {result["p1_identity_score"]*W_P1:.4f}')
        print(f'  │ P2 Crime Severity  : {result["p2_crime_severity"]:.4f}  × {W_P2} = {result["p2_crime_severity"]*W_P2:.4f}')
        print(f'  │ P4 Linkage Score   : {result["p4_linkage_score"]:.4f}  × {W_P4} = {result["p4_linkage_score"]*W_P4:.4f}')
        print(f'  │ P5 Visual Score    : {result["p5_visual_score"]:.4f}  × {W_P5} = {result["p5_visual_score"]*W_P5:.4f}')
        print(f'  ├─────────────────────────────────────────────────────────────')
        print(f'  │ FINAL RISK SCORE   : {result["final_risk"]:.4f}')
        print(f'  └─────────────────────────────────────────────────────────────')
        print(f'  Action: {result["action"]}')
    
    # P4 Breakdown
    if result['p4_breakdown']:
        print(f'  P4 Linkage Breakdown:')
        for link in result['p4_breakdown']:
            print(f'    Rank {link["rank"]} ({link["rank_weight"]:.0%}): '
                  f'{link["candidate_name"]} — link={link["link_score"]:.4f}, '
                  f'crime={link["crime_type"]} (sev={link["crime_severity"]}), '
                  f'contribution={link["contribution"]:.4f}')
    else:
        print(f'  P4 Linkage: No pre-computed links available')
    
    # Top-3 identity matches
    print(f'  P1 Top-3 Matches:')
    for rank, (name, sc) in enumerate(result['top3_matches'], 1):
        flag = '🔴' if sc >= 0.75 else '🟡' if sc >= 0.50 else '  '
        print(f'    #{rank} {name}: {sc:.4f} {flag}')

  INTEGRATED RISK SCORING PIPELINE
  Formula: P1(0.40) + P2(0.30) + P4(0.20) + P5(0.10)

🔴 CRITICAL — Client: "S. Nikitenko" (Initial + surname)
  Matched Fugitive : SANAVBARI NIKITENKO [NK-224TRezPqwzhQZ37exWxtX]
  Matched Crime    : terrorism
  ┌─────────────────────────────────────────────────────────────
  │ P3 Biometric Conf. : N/A  (Client not in biometric database)
  │ P1 Identity Score  : 0.7624  × 0.4 = 0.3050
  │ P2 Crime Severity  : 1.0000  × 0.3 = 0.3000
  │ P4 Linkage Score   : 0.8342  × 0.2 = 0.1668
  │ P5 Visual Score    : 0.5000  × 0.1 = 0.0500
  ├─────────────────────────────────────────────────────────────
  │ FINAL RISK SCORE   : 0.8218
  └─────────────────────────────────────────────────────────────
  Action: Immediate escalation — freeze transaction, notify compliance officer
  P4 Linkage Breakdown:
    Rank 1 (50%): ZAUR GULIEV — link=1.0000, crime=terrorism (sev=1.0), contribution=0.5000
    Rank 2 (30%): VALEH AMIRALIEV — link=0.9938, crime=armed formation (sev=

## 5. Summary Table

In [28]:
# ═════════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ═════════════════════════════════════════════════════════════════════════════

summary_data = []
for r in all_results:
    summary_data.append({
        'Client Query': r['client_name'],
        'Matched Fugitive': r['matched_name'],
        'Crime Type': r['matched_crime'],
        'P3 Bio': r.get('p3_confidence', 'N/A'),
        'P1 (40%)': r['p1_identity_score'],
        'P2 (30%)': r['p2_crime_severity'],
        'P4 (20%)': r['p4_linkage_score'],
        'P5 (10%)': r['p5_visual_score'],
        'Final Risk': r['final_risk'],
        'Tier': f'{r["icon"]} {r["tier"]}',
    })

df_summary = pd.DataFrame(summary_data)

print('\n' + '=' * 90)
print('  SCREENING RESULTS SUMMARY')
print('=' * 90)
print(df_summary.to_string(index=False))


  SCREENING RESULTS SUMMARY
         Client Query              Matched Fugitive      Crime Type  P3 Bio  P1 (40%)  P2 (30%)  P4 (20%)  P5 (10%)  Final Risk        Tier
         S. Nikitenko           SANAVBARI NIKITENKO       terrorism     NaN    0.7624       1.0    0.8342       0.5      0.8218  🔴 CRITICAL
         Hasen Aksema          HASEN AMHAMED AKSEMA        homicide     NaN    0.8626       0.9    0.5550       0.5      0.7760  🔴 CRITICAL
     Abdul Sambolotov              ABDUL SAMBOLATOV       terrorism     NaN    0.7486       1.0    0.5543       0.5      0.7603  🔴 CRITICAL
             JIAN XIA                      JIAN XIA        homicide  0.0285    1.0000       0.0    0.0000       0.0      0.0000  ⬜ LOW RISK
       Norbert Bialas                NORBERT BIALAS         assault  0.0002    1.0000       0.0    0.0000       0.0      0.0000  ⬜ LOW RISK
     Ramazan Chigayev               RAMAZAN CHIGAEV armed formation     NaN    0.7914       0.7    0.7000       0.5      0.7166  🔴 

In [29]:
# ── Tier Distribution ─────────────────────────────────────────────────────────
from collections import Counter

print('\n' + '─' * 60)
print('  TIER DISTRIBUTION')
print('─' * 60)

tier_counts = Counter(r['tier'] for r in all_results)
for tier_name in ['CRITICAL', 'HIGH RISK', 'REVIEW', 'LOW RISK']:
    count = tier_counts.get(tier_name, 0)
    bar = '█' * count
    print(f'  {tier_name:<12} {bar}  ({count})')

print(f'\n  Total screened: {len(all_results)}')


────────────────────────────────────────────────────────────
  TIER DISTRIBUTION
────────────────────────────────────────────────────────────
  CRITICAL     █████████████  (13)
  HIGH RISK    ██  (2)
  REVIEW         (0)
  LOW RISK     ██  (2)

  Total screened: 17


In [30]:
# ── Threshold Reference ───────────────────────────────────────────────────────
print('\n' + '─' * 60)
print('  RISK THRESHOLD REFERENCE')
print('─' * 60)
print(f'  {"Tier":<12} {"Threshold":>10}  Action')
print('  ' + '─' * 55)

tier_actions = {
    'CRITICAL':  'Freeze + escalate to compliance officer',
    'HIGH RISK': 'Senior analyst review within 24hr',
    'REVIEW':    'Junior analyst flag, monitor 30 days',
    'LOW RISK':  'Pass, log to audit trail',
}
for tier_name, threshold in FINAL_THRESHOLDS.items():
    print(f'  {tier_name:<12} {threshold:>10.2f}  {tier_actions[tier_name]}')

print(f'\n  Formula: P1×{W_P1} + P2×{W_P2} + P4×{W_P4} + P5×{W_P5}')
print(f'  P4 = Sum of (rank_weight_k / weight_sum) × link_score_k × severity_k')
print(f'       Rank weights normalised so they sum to 1.0 for available ranks (0.0 - 1.0)')


────────────────────────────────────────────────────────────
  RISK THRESHOLD REFERENCE
────────────────────────────────────────────────────────────
  Tier          Threshold  Action
  ───────────────────────────────────────────────────────
  CRITICAL           0.70  Freeze + escalate to compliance officer
  HIGH RISK          0.50  Senior analyst review within 24hr
  REVIEW             0.30  Junior analyst flag, monitor 30 days
  LOW RISK           0.00  Pass, log to audit trail

  Formula: P1×0.4 + P2×0.3 + P4×0.2 + P5×0.1
  P4 = Sum of (rank_weight_k / weight_sum) × link_score_k × severity_k
       Rank weights normalised so they sum to 1.0 for available ranks (0.0 - 1.0)


## 6. Export Results

In [31]:
# ═════════════════════════════════════════════════════════════════════════════
# EXPORT
# ═════════════════════════════════════════════════════════════════════════════

export_data = []
for r in all_results:
    row = {
        'client_query': r['client_name'],
        'matched_fugitive_id': r['matched_id'],
        'matched_fugitive_name': r['matched_name'],
        'matched_crime_type': r['matched_crime'],
        'p3_biometric_confidence': r.get('p3_confidence'),
        'p3_biometric_match': r.get('p3_match'),
        'p1_identity_score': r['p1_identity_score'],
        'p2_crime_severity': r['p2_crime_severity'],
        'p4_linkage_score': r['p4_linkage_score'],
        'p5_visual_score': r['p5_visual_score'],
        'final_risk_score': r['final_risk'],
        'risk_tier': r['tier'],
        'action': r['action'],
    }
    
    # Add P4 breakdown columns
    for link in r['p4_breakdown']:
        rank = link['rank']
        row[f'p4_rank{rank}_fugitive'] = link['candidate_name']
        row[f'p4_rank{rank}_link_score'] = link['link_score']
        row[f'p4_rank{rank}_crime'] = link['crime_type']
        row[f'p4_rank{rank}_severity'] = link['crime_severity']
        row[f'p4_rank{rank}_contribution'] = link['contribution']
    
    export_data.append(row)

df_export = pd.DataFrame(export_data)
EXPORT_PATH = os.path.join(OUTPUT_DIR, 'integrated_screening_results.csv')
df_export.to_csv(EXPORT_PATH, index=False)
print(f'✅ Results exported to: {EXPORT_PATH}')
print(f'   Rows: {len(df_export)}, Columns: {len(df_export.columns)}')

✅ Results exported to: /Volumes/My Mini Drive/CS 610/CS610_Interpol/Integration/outputs/integrated_screening_results.csv
   Rows: 17, Columns: 28


## 7. RAG Chatbot — Streamlit + ChatGPT

Generates a standalone Streamlit app (`streamlit_app.py`) that:
- Loads the integration pipeline artifacts (P1 pickle, P2 CSV, P4 CSV)
- Lets users enter their OpenAI API key in the sidebar
- Supports two modes via natural language:
  - **Screen a client** — runs the full risk pipeline and explains results
  - **Explore the database** — answers questions about fugitives, crime stats, clusters
- Builds RAG context from pipeline outputs and sends to ChatGPT for natural-language explanation

**Run with:**
```bash
pip install streamlit openai
cd Integration
streamlit run streamlit_app.py
```

In [32]:
STREAMLIT_CODE = r'''
import os, pickle, re, json
from datetime import datetime
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import streamlit as st
from dateutil.relativedelta import relativedelta
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
W_P1, W_P2, W_P4, W_P5 = 0.40, 0.30, 0.20, 0.10
LINK_RANK_WEIGHTS = {1: 0.50, 2: 0.30, 3: 0.20}
DEFAULT_VISUAL_SCORE = 0.5
FINAL_THRESHOLDS = {'CRITICAL': 0.70, 'HIGH RISK': 0.50, 'REVIEW': 0.30, 'LOW RISK': 0.00}
P3_THRESHOLD = 0.5

SYSTEM_PROMPT = """You are an AI analyst for the INTERPOL Risk Intelligence System (CS610 project).
You help compliance officers understand screening results and explore the fugitive database.

PIPELINE OVERVIEW (P3 is a biometric confirmation check):
- Pillar 3 (confirmation): Biometric prediction via ensemble voting (LR + RF + XGBoost). Compares client vs matched fugitive on 7 features: name_similarity, age_difference, same_gender, height_difference, weight_difference, same_hair_colour, same_eye_colour. Outputs confidence (0-1). If >= 0.5 (MATCH), the identity is biometrically confirmed → immediate CRITICAL escalation. If < 0.5 (MISMATCH), biometrics are inconclusive → proceed with full multi-pillar scoring.
- Pillar 1 (40%): TF-IDF char n-gram cosine similarity for identity resolution
- Pillar 2 (30%): Crime severity from categorisation export (terrorism=1.0, homicide=0.9, sexual crime=0.8, armed formation/assault/narcotics=0.7, financial crime=0.5)
- Pillar 4 (20%): Hidden linkage from GraphSAGE link predictions (top-3 linked fugitives, rank weights normalised to sum to 1.0)
- Pillar 5 (10%): Visual similarity via CLIP (placeholder 0.5 until delivered)
- If P3 MATCH: Final Risk = 1.0 (CRITICAL). If P3 mismatch/N/A: Final Risk = P1×0.40 + P2×0.30 + P4×0.20 + P5×0.10

RISK TIERS:
- CRITICAL (≥0.70): Freeze + escalate to compliance officer
- HIGH RISK (≥0.50): Senior analyst review within 24hr
- REVIEW (≥0.30): Junior analyst flag, monitor 30 days
- LOW RISK (<0.30): Pass, log to audit trail

BIOMETRIC DATA COLLECTION:
When a user requests to screen a client but only provides a name, you MUST ask for the following biometric details before screening can proceed:
1. Gender (M/F)
2. Date of birth (or approximate age)
3. Height (in metres, e.g. 1.75)
4. Weight (in kg, e.g. 70)
5. Hair colour (e.g. Black, Brown, Blond, Red, Grey, White)
6. Eye colour (e.g. Brown, Blue, Green, Hazel, Grey)

Present this as a friendly form-like request. Explain that biometric data enables the P3 biometric confirmation check — if biometrics match, the case is immediately escalated as CRITICAL; if they don't match, the system proceeds with full multi-pillar evaluation.

If the user provides ALL biometric fields along with the name in one message, proceed directly with screening.

When you receive the biometric data, respond with EXACTLY this format on its own line so the system can parse it:
BIOMETRICS_COLLECTED: gender=<M or F>, dob=<YYYY-MM-DD or age_XX>, height=<metres>, weight=<kg>, hair=<colour>, eye=<colour>

RESPONSE FORMAT RULES — you MUST follow these:
- Use markdown formatting: **bold** for emphasis, headers (##, ###) for sections, bullet points for lists.
- For screening results, structure your response with these sections:
  ### Match Summary
  ### Pillar Breakdown
  ### Risk Assessment
  ### Recommended Action
- Use tables when comparing data (e.g. pillar scores, fugitive comparisons).
- Keep responses concise and professional — 150-300 words for screening, shorter for DB queries.
- Use the data provided — do not fabricate scores or fugitive names."""

# ─────────────────────────────────────────────────────────────────────────────
# DATA LOADING (cached)
# ─────────────────────────────────────────────────────────────────────────────
@st.cache_resource
def load_pipeline():
    # ── Directory config (mirrors notebook Cell 4) ────────────────────────────
    repo_root  = os.path.abspath(os.path.join(os.getcwd(), '..'))

    def _res(*candidates):
        for p in candidates:
            if os.path.exists(p):
                return p
        return candidates[-1]

    p1_base    = os.path.join(repo_root, 'Pillar 1 (Identity Resolution)')
    p2_base    = os.path.join(repo_root, 'Pillar 2 (Crime Severity)')
    p3_base    = os.path.join(repo_root, 'Pillar 3 (Biometric Prediction)')
    p4_base    = os.path.join(repo_root, 'Pillar 4 (GCN and Graphsage)')
    p4_out_dir = os.path.join(p4_base, 'outputs')

    fugitive_csv = os.path.join(p4_base, 'crime_analysis_results_aft_transformer_ner.csv')

    p1_vectorizer_pkl = _res(
        os.path.join(p1_base, 'outputs', 'pillar1_name_vectorizer.pkl'),
        os.path.join(p1_base, 'output',  'pillar1_name_vectorizer.pkl'),
        os.path.join(repo_root, 'outputs', 'pillar1_name_vectorizer.pkl'),
    )
    p1_embeddings_pkl = _res(
        os.path.join(p1_base, 'outputs', 'pillar1_name_embeddings.pkl'),
        os.path.join(p1_base, 'output',  'pillar1_name_embeddings.pkl'),
        os.path.join(repo_root, 'outputs', 'pillar1_name_embeddings.pkl'),
    )
    p2_crime_csv = _res(
        os.path.join(p2_base, 'crime_categorisation_export.csv'),
        os.path.join(p2_base, 'outputs', 'crime_categorisation_export.csv'),
    )
    p4_links_csv = os.path.join(p4_out_dir, 'top_new_links_all.csv')

    # P2: Crime categorisation
    df_crime = pd.read_csv(p2_crime_csv)
    severity_map = df_crime.set_index('id')[['final_crime_label', 'severity_score', 'risk_tier']].to_dict('index')
    crime_severity = df_crime.groupby('final_crime_label')['severity_score'].first().to_dict()
    crime_severity['Unknown'] = 0.1

    # Main fugitive CSV
    df_fug = pd.read_csv(fugitive_csv)
    df_unique = df_fug.drop_duplicates(subset='id', keep='first').reset_index(drop=True)
    df_unique['final_crime_label'] = df_unique['id'].map(
        lambda x: severity_map.get(x, {}).get('final_crime_label', 'Unknown'))
    df_unique['severity_score'] = df_unique['id'].map(
        lambda x: severity_map.get(x, {}).get('severity_score', 0.1))

    # P1: TF-IDF from pickle or rebuild
    if os.path.exists(p1_vectorizer_pkl) and os.path.exists(p1_embeddings_pkl):
        with open(p1_vectorizer_pkl, 'rb') as f:
            vectorizer = pickle.load(f)
        with open(p1_embeddings_pkl, 'rb') as f:
            embeddings = pickle.load(f)
    else:
        vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4),
                                     max_features=5000, sublinear_tf=True)
        embeddings = normalize(vectorizer.fit_transform(df_unique['name'].str.upper().astype(str)))

    # P4: Links CSV + hidden linkage scores
    df_links = pd.read_csv(p4_links_csv) if os.path.exists(p4_links_csv) else pd.DataFrame()
    p4_scores_csv = os.path.join(p4_out_dir, 'hidden_linkage_scores.csv')
    if os.path.exists(p4_scores_csv):
        linkage_score_map = pd.read_csv(p4_scores_csv).set_index('id')['linkage_score'].to_dict()
    else:
        linkage_score_map = {}

    # P3: Biometric model + client database
    p3_model_pkl = _res(
        os.path.join(repo_root, 'outputs', 'pillar3_ensemble_voting.pkl'),
        os.path.join(p3_base, 'output', 'ensemble_voting.pkl'),
        os.path.join(p3_base, 'outputs', 'ensemble_voting.pkl'),
    )
    p3_model = None
    if os.path.exists(p3_model_pkl):
        with open(p3_model_pkl, 'rb') as f:
            p3_model = pickle.load(f)

    client_csv = os.path.join(repo_root, 'synthetic_data_generation', 'synthetic_client_data.csv')
    client_bio_map = {}
    if os.path.exists(client_csv):
        df_cl = pd.read_csv(client_csv)
        df_cl['full_name_upper'] = df_cl['full_name'].str.upper().str.strip()
        df_cl = df_cl.drop_duplicates(subset='full_name_upper', keep='first')
        client_bio_map = df_cl.set_index('full_name_upper').to_dict('index')

    return {
        'df_unique': df_unique, 'vectorizer': vectorizer, 'embeddings': embeddings,
        'df_links': df_links, 'linkage_score_map': linkage_score_map,
        'severity_map': severity_map, 'crime_severity': crime_severity,
        'df_crime': df_crime,
        'p3_model': p3_model, 'client_bio_map': client_bio_map,
    }

# ─────────────────────────────────────────────────────────────────────────────
# PILLAR 3 HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def _parse_dob_st(val):
    if pd.isna(val) or str(val).strip() in ('', 'nan', 'NaN'):
        return None
    try:
        return pd.to_datetime(str(val).strip())
    except Exception:
        return None

def compute_p3_features_st(client, fugitive):
    c_name = str(client.get('full_name', ''))
    f_name = str(fugitive.get('name', ''))
    name_sim = SequenceMatcher(None, c_name.upper(), f_name.upper()).ratio()
    today = datetime.today()
    c_dob, f_dob = _parse_dob_st(client.get('date_of_birth')), _parse_dob_st(fugitive.get('birth_date'))
    age_diff = abs(relativedelta(today, c_dob).years - relativedelta(today, f_dob).years) if c_dob and f_dob else 0
    c_g = str(client.get('gender', '')).strip().upper()
    f_g = str(fugitive.get('GENDER', '')).strip().upper()
    same_gender = 1 if (c_g and f_g and c_g == f_g) else 0
    try: c_h = float(client.get('height_m', 0) or 0)
    except: c_h = 0.0
    try: f_h = float(fugitive.get('height', 0) or 0)
    except: f_h = 0.0
    height_diff = abs(c_h - f_h)
    try: c_w = float(client.get('weight_kg', 0) or 0)
    except: c_w = 0.0
    try: f_w = float(fugitive.get('weight', 0) or 0)
    except: f_w = 0.0
    weight_diff = abs(c_w - f_w)
    c_hair = str(client.get('hair_color', '')).strip().upper()
    f_hair = str(fugitive.get('hairColor', '')).strip().upper()
    same_hair = 1 if (c_hair and f_hair and c_hair not in ('', 'NAN') and f_hair not in ('', 'NAN', 'OTHD') and c_hair == f_hair) else 0
    c_eye = str(client.get('eye_color', '')).strip().upper()
    f_eye = str(fugitive.get('eyeColor', '')).strip().upper()
    same_eye = 1 if (c_eye and f_eye and c_eye not in ('', 'NAN') and f_eye not in ('', 'NAN', 'OTHD') and c_eye == f_eye) else 0
    return [name_sim, age_diff, same_gender, height_diff, weight_diff, same_hair, same_eye]

# ─────────────────────────────────────────────────────────────────────────────
# PIPELINE FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────
def screen(client_name, data, client_bio=None):
    df = data['df_unique']
    # Candidate retrieval via TF-IDF
    q = normalize(data['vectorizer'].transform([client_name.upper()]))
    sims = cosine_similarity(q, data['embeddings'])[0]
    top_idx = int(np.argmax(sims))
    p1_score = float(sims[top_idx])
    fug = df.iloc[top_idx]
    fid = fug['id']

    top3_idx = np.argsort(sims)[::-1][:3]
    top3 = [(df.iloc[i]['name'], round(float(sims[i]), 4)) for i in top3_idx]

    # P3: Biometric validation (runs FIRST)
    p3_conf, p3_match_flag, p3_note = None, None, ''
    p3_model = data.get('p3_model')
    client_bio_map = data.get('client_bio_map', {})
    client_key = client_name.upper().strip()
    bio = client_bio if client_bio else client_bio_map.get(client_key)
    if p3_model and bio:
        if 'full_name' not in bio:
            bio['full_name'] = client_name
        feats = compute_p3_features_st(bio, fug)
        feats = [0.0 if (v is None or (isinstance(v, float) and np.isnan(v))) else v for v in feats]
        p3_proba = p3_model.predict_proba([feats])[0]
        p3_conf = float(p3_proba[1])
        p3_match_flag = int(p3_conf >= P3_THRESHOLD)
    else:
        p3_note = 'No biometric data provided' if p3_model else 'P3 model not loaded'

    p3_str = f'{p3_conf:.4f}' if p3_conf is not None else f'N/A ({p3_note})'
    p3_verdict = 'MATCH' if p3_match_flag == 1 else ('NO MATCH' if p3_match_flag == 0 else 'N/A')
    p3_confirmed = (p3_match_flag == 1) if p3_match_flag is not None else False

    if p3_confirmed:
        # Biometrics confirm identity → strongest evidence → immediate CRITICAL
        if fid in data['severity_map']:
            e = data['severity_map'][fid]
            p2_label = e['final_crime_label']
        else:
            p2_label = fug.get('final_crime_label', 'Unknown')
        ctx = f"""SCREENING RESULT FOR: "{client_name}"
Matched Fugitive: {fug['name']} [{fid}]
Crime Type: {p2_label}

P3 BIOMETRIC CONFIRMATION:
  Confidence: {p3_str}  ({p3_verdict})
  Biometric match confirmed — this is the strongest evidence.

FINAL RISK SCORE: 1.0000
RISK TIER: CRITICAL
ACTION: Biometric match confirmed — freeze + escalate immediately

TOP-3 IDENTITY MATCHES:
""" + "\\n".join(f"  #{i+1} {n}: {s:.4f}" for i, (n, s) in enumerate(top3))
        return ctx, 'CRITICAL', 1.0

    # P3 mismatch or N/A → need all pillars to evaluate
    if fid in data['severity_map']:
        e = data['severity_map'][fid]
        p2_label, p2_score = e['final_crime_label'], e['severity_score']
    else:
        ct = fug.get('final_crime_label', 'Unknown')
        p2_label = ct
        p2_score = data['crime_severity'].get(ct, 0.1)

    links = data['df_links']
    link_rows = links[links['anchor_id'] == fid].sort_values('rank') if not links.empty else pd.DataFrame()
    avail_ranks = [int(r['rank']) for _, r in link_rows.iterrows()]
    wsum = sum(LINK_RANK_WEIGHTS.get(r, 0.0) for r in avail_ranks)
    p4_total, p4_detail = 0.0, []
    for _, row in link_rows.iterrows():
        rank = int(row['rank'])
        ls = float(row['predicted_link_score'])
        cid = row['candidate_id']
        if cid in data['severity_map']:
            cl, cs = data['severity_map'][cid]['final_crime_label'], data['severity_map'][cid]['severity_score']
        else:
            cl, cs = 'Unknown', 0.1
        rw = LINK_RANK_WEIGHTS.get(rank, 0.0)
        nw = rw / wsum if wsum > 0 else 0.0
        contrib = nw * ls * cs
        p4_total += contrib
        p4_detail.append(f"  Rank {rank} ({nw:.0%}): {row['candidate_name']} — link={ls:.4f}, crime={cl} (sev={cs}), contrib={contrib:.4f}")
    if link_rows.empty and fid in data.get('linkage_score_map', {}):
        p4_total = data['linkage_score_map'][fid]
        p4_detail.append(f"  (score from hidden_linkage_scores.csv — no per-link detail)")

    p5_score = DEFAULT_VISUAL_SCORE
    final = p1_score * W_P1 + p2_score * W_P2 + p4_total * W_P4 + p5_score * W_P5

    if final >= 0.70: tier, action = 'CRITICAL', 'Freeze + escalate to compliance officer'
    elif final >= 0.50: tier, action = 'HIGH RISK', 'Senior analyst review within 24hr'
    elif final >= 0.30: tier, action = 'REVIEW', 'Junior analyst flag, monitor 30 days'
    else: tier, action = 'LOW RISK', 'Pass, log to audit trail'

    ctx = f"""SCREENING RESULT FOR: "{client_name}"
Matched Fugitive: {fug['name']} [{fid}]
Crime Type: {p2_label}

PILLAR SCORES:
  P3 Biometric Conf.:    {p3_str}  ({p3_verdict})  <- full evaluation needed
  P1 Identity (TF-IDF): {p1_score:.4f} × {W_P1} = {p1_score*W_P1:.4f}
  P2 Crime Severity:     {p2_score:.4f} × {W_P2} = {p2_score*W_P2:.4f}
  P4 Hidden Linkage:     {p4_total:.4f} × {W_P4} = {p4_total*W_P4:.4f}
  P5 Visual Similarity:  {p5_score:.4f} × {W_P5} = {p5_score*W_P5:.4f}

FINAL RISK SCORE: {final:.4f}
RISK TIER: {tier}
ACTION: {action}

TOP-3 IDENTITY MATCHES:
""" + "\n".join(f"  #{i+1} {n}: {s:.4f}" for i, (n, s) in enumerate(top3))

    if p4_detail:
        ctx += "\n\nP4 LINKAGE BREAKDOWN:\n" + "\n".join(p4_detail)
    else:
        ctx += "\n\nP4 LINKAGE: No pre-computed links for this fugitive"

    return ctx, tier, final


def db_context(query, data):
    df = data['df_unique']
    parts = []

    parts.append(f"DATABASE SUMMARY: {len(df):,} unique fugitives")
    crime_dist = df['final_crime_label'].value_counts()
    parts.append("CRIME DISTRIBUTION:\n" + "\n".join(f"  {k}: {v:,}" for k, v in crime_dist.items()))

    if 'GENDER' in df.columns:
        gender_dist = df['GENDER'].value_counts()
        parts.append("GENDER: " + ", ".join(f"{k}={v:,}" for k, v in gender_dist.items()))

    if 'age_today' in df.columns:
        ages = df['age_today'].dropna()
        parts.append(f"AGE: mean={ages.mean():.1f}, min={ages.min():.0f}, max={ages.max():.0f}")

    if 'label' in df.columns:
        top_countries = df['label'].value_counts().head(10)
        parts.append("TOP 10 COUNTRIES:\n" + "\n".join(f"  {k}: {v:,}" for k, v in top_countries.items()))

    name_matches = df[df['name'].str.contains(query.upper(), na=False)]
    if not name_matches.empty and len(name_matches) <= 20:
        rows = []
        for _, r in name_matches.head(10).iterrows():
            rows.append(f"  {r['name']} [{r['id']}] — {r.get('final_crime_label', '?')}, "
                        f"age={r.get('age_today', '?')}, country={r.get('label', '?')}")
        parts.append(f"NAME MATCHES FOR '{query}':\n" + "\n".join(rows))

    severity_dist = df.groupby('final_crime_label')['severity_score'].first()
    parts.append("SEVERITY SCORES:\n" + "\n".join(f"  {k}: {v}" for k, v in severity_dist.items()))

    links = data['df_links']
    if not links.empty:
        parts.append(f"P4 LINKAGE DATA: {len(links):,} link predictions across {links['anchor_id'].nunique():,} anchors")
    lsm = data.get('linkage_score_map', {})
    if lsm:
        parts.append(f"P4 HIDDEN LINKAGE SCORES: {len(lsm):,} fugitives with pre-computed mean scores")

    return "\n\n".join(parts)

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
TIER_STYLES = {
    'CRITICAL':  {'bg': '#fef2f2', 'border': '#fca5a5', 'color': '#991b1b', 'icon': '\U0001f534', 'bar': '#dc2626'},
    'HIGH RISK': {'bg': '#fff7ed', 'border': '#fdba74', 'color': '#9a3412', 'icon': '\U0001f7e0', 'bar': '#ea580c'},
    'REVIEW':    {'bg': '#fefce8', 'border': '#fde047', 'color': '#854d0e', 'icon': '\U0001f7e1', 'bar': '#ca8a04'},
    'LOW RISK':  {'bg': '#f0fdf4', 'border': '#86efac', 'color': '#166534', 'icon': '\u2705', 'bar': '#16a34a'},
}

def parse_biometrics(text):
    """Extract BIOMETRICS_COLLECTED line from LLM response and build a client_bio dict."""
    for line in text.splitlines():
        line = line.strip()
        if line.startswith('BIOMETRICS_COLLECTED:'):
            parts = line.split(':', 1)[1].strip()
            bio = {}
            for pair in parts.split(','):
                pair = pair.strip()
                if '=' not in pair:
                    continue
                k, v = pair.split('=', 1)
                k, v = k.strip().lower(), v.strip()
                if k == 'gender':
                    bio['gender'] = v.upper()
                elif k == 'dob':
                    if v.startswith('age_'):
                        try:
                            age = int(v.replace('age_', ''))
                            from datetime import datetime, timedelta
                            bio['date_of_birth'] = (datetime.today() - timedelta(days=age*365)).strftime('%Y-%m-%d')
                        except: pass
                    else:
                        bio['date_of_birth'] = v
                elif k == 'height':
                    try: bio['height_m'] = float(v)
                    except: pass
                elif k == 'weight':
                    try: bio['weight_kg'] = float(v)
                    except: pass
                elif k == 'hair':
                    bio['hair_color'] = v
                elif k == 'eye':
                    bio['eye_color'] = v
            return bio if bio else None
    return None


def render_score_card(client_name, tier, score, rag_context):
    s = TIER_STYLES.get(tier, TIER_STYLES['REVIEW'])
    pct = min(score * 100, 100)
    lines = rag_context.split('\n')
    matched = next((l for l in lines if l.startswith('Matched Fugitive:')), '')
    crime = next((l for l in lines if l.startswith('Crime Type:')), '')
    p1_line = next((l for l in lines if 'P1 Identity' in l), '')
    p2_line = next((l for l in lines if 'P2 Crime' in l), '')
    p4_line = next((l for l in lines if 'P4 Hidden' in l), '')
    p5_line = next((l for l in lines if 'P5 Visual' in l), '')
    def extract_scores(line):
        parts = line.strip().split()
        raw = parts[-3] if len(parts) >= 3 else '\u2014'
        weighted = parts[-1] if len(parts) >= 1 else '\u2014'
        return raw, weighted
    p1r, p1w = extract_scores(p1_line)
    p2r, p2w = extract_scores(p2_line)
    p4r, p4w = extract_scores(p4_line)
    p5r, p5w = extract_scores(p5_line)
    return f"""<div style="background:{s['bg']}; border:1px solid {s['border']}; border-radius:12px;
                padding:1.2rem 1.5rem; margin-bottom:1rem; font-family:'Inter',sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:0.8rem;">
    <div>
      <div style="font-size:0.7rem; text-transform:uppercase; letter-spacing:0.08em; color:{s['color']}; font-weight:700;">{s['icon']} {tier}</div>
      <div style="font-size:1.3rem; font-weight:700; color:#1a1a2e;">Screening: {client_name}</div>
    </div>
    <div style="text-align:right;">
      <div style="font-size:2rem; font-weight:800; color:{s['color']}; line-height:1;">{score:.2f}</div>
      <div style="font-size:0.65rem; color:#64748b; text-transform:uppercase;">Risk Score</div>
    </div>
  </div>
  <div style="background:#e2e8f0; border-radius:6px; height:8px; margin-bottom:1rem; overflow:hidden;">
    <div style="background:{s['bar']}; height:100%; width:{pct}%; border-radius:6px;"></div>
  </div>
  <div style="display:grid; grid-template-columns:1fr 1fr; gap:0.4rem 1.5rem; font-size:0.82rem; color:#334155;">
    <div><span style="color:#64748b;">Matched:</span> <b>{matched.replace('Matched Fugitive: ','')}</b></div>
    <div><span style="color:#64748b;">Crime:</span> <b>{crime.replace('Crime Type: ','')}</b></div>
    <div><span style="color:#64748b;">P1 Identity:</span> <code>{p1r}</code> &rarr; <b>{p1w}</b></div>
    <div><span style="color:#64748b;">P2 Severity:</span> <code>{p2r}</code> &rarr; <b>{p2w}</b></div>
    <div><span style="color:#64748b;">P4 Linkage:</span> <code>{p4r}</code> &rarr; <b>{p4w}</b></div>
    <div><span style="color:#64748b;">P5 Visual:</span> <code>{p5r}</code> &rarr; <b>{p5w}</b></div>
  </div>
</div>"""

# ─────────────────────────────────────────────────────────────────────────────
# STREAMLIT UI
# ─────────────────────────────────────────────────────────────────────────────
st.set_page_config(page_title="INTERPOL Risk Intelligence", page_icon="\U0001f534", layout="centered")

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

    .stApp {
        background: linear-gradient(135deg, #f8f9fc 0%, #eef1f8 100%);
    }
    .block-container { max-width: 860px; padding-top: 2rem; }

    [data-testid="stSidebar"] {
        background: linear-gradient(180deg, #0f1629 0%, #1a2342 100%);
    }
    [data-testid="stSidebar"] * {
        color: #c8cfe0 !important;
    }
    [data-testid="stSidebar"] h1,
    [data-testid="stSidebar"] h2,
    [data-testid="stSidebar"] h3 {
        color: #ffffff !important;
    }
    [data-testid="stSidebar"] .stDivider {
        border-color: rgba(255,255,255,0.1);
    }

    .header-bar {
        background: linear-gradient(135deg, #0f1629 0%, #1e2a4a 100%);
        border-radius: 12px;
        padding: 1.8rem 2rem;
        margin-bottom: 1.5rem;
        border: 1px solid rgba(255,255,255,0.06);
    }
    .header-bar h1 {
        color: #ffffff;
        font-family: 'Inter', sans-serif;
        font-weight: 700;
        font-size: 1.7rem;
        margin: 0 0 0.3rem 0;
        letter-spacing: -0.02em;
    }
    .header-bar p {
        color: #8b95b0;
        font-size: 0.85rem;
        margin: 0;
    }
    .header-bar .accent {
        color: #e74c3c;
        font-weight: 600;
    }

    .pillar-row {
        display: flex; gap: 0.6rem; margin-top: 0.8rem; flex-wrap: wrap;
    }
    .pillar-tag {
        font-size: 0.7rem;
        padding: 0.25rem 0.6rem;
        border-radius: 20px;
        font-weight: 600;
        font-family: 'Inter', sans-serif;
        letter-spacing: 0.02em;
    }
    .pillar-tag.p1 { background: #1e3a5f; color: #7eb8f7; }
    .pillar-tag.p2 { background: #3a1f1f; color: #f7a07e; }
    .pillar-tag.p4 { background: #1f3a2a; color: #7ef7a0; }
    .pillar-tag.p5 { background: #3a2a1f; color: #f7d77e; }

    .sidebar-example {
        background: rgba(255,255,255,0.06);
        border: 1px solid rgba(255,255,255,0.08);
        border-radius: 8px;
        padding: 0.5rem 0.75rem;
        margin: 0.3rem 0;
        font-family: 'Inter', monospace;
        font-size: 0.8rem;
        color: #a0b0cc !important;
        cursor: default;
        transition: background 0.2s;
    }
    .sidebar-example:hover { background: rgba(255,255,255,0.1); }
    .sidebar-label {
        font-size: 0.7rem;
        text-transform: uppercase;
        letter-spacing: 0.08em;
        color: #6b7a96 !important;
        font-weight: 600;
        margin: 1rem 0 0.5rem 0;
    }

    [data-testid="stChatMessage"] {
        background: #ffffff;
        border: 1px solid #e2e8f0;
        border-radius: 12px;
        padding: 1rem 1.2rem;
        margin-bottom: 0.8rem;
        box-shadow: 0 1px 3px rgba(0,0,0,0.04);
    }
    [data-testid="stChatInput"] textarea {
        border-radius: 12px !important;
        border: 2px solid #d0d7e3 !important;
        background: #ffffff !important;
        font-size: 0.95rem !important;
    }
    [data-testid="stChatInput"] textarea:focus {
        border-color: #4a6cf7 !important;
        box-shadow: 0 0 0 3px rgba(74,108,247,0.15) !important;
    }
</style>
""", unsafe_allow_html=True)

with st.sidebar:
    st.markdown("""
    <div style="text-align:center; padding: 0.5rem 0 0.2rem 0;">
        <div style="font-size:2rem;">🔴</div>
        <div style="font-size:1.2rem; font-weight:700; color:#fff !important; letter-spacing:0.04em;">
            INTERPOL IRIS
        </div>
        <div style="font-size:0.75rem; color:#6b7a96 !important; margin-top:0.15rem;">
            Integrated Risk Intelligence System
        </div>
    </div>
    """, unsafe_allow_html=True)
    st.divider()
    api_key = st.text_input("OpenAI API Key", type="password", placeholder="sk-...")
    model = "gpt-5-mini"
    st.markdown(f'<div style="font-size:0.8rem; color:#8b95b0 !important; margin-top:0.3rem;">Model: <b style="color:#fff !important;">{model}</b></div>', unsafe_allow_html=True)
    st.divider()
    st.markdown('<div class="sidebar-label">Example queries</div>', unsafe_allow_html=True)
    for ex in [
        "Screen S. Nikitenko",
        "Screen Abdul Sambolotov",
        "How many terrorism fugitives?",
        "Show me fugitives from Russia",
        "What crime types are in the database?",
    ]:
        st.markdown(f'<div class="sidebar-example">{ex}</div>', unsafe_allow_html=True)

st.markdown("""
<div class="header-bar">
    <h1>INTERPOL Risk Intelligence <span class="accent">Chatbot</span></h1>
    <p>CS610 Integrated Screening Pipeline &mdash; ask a question or screen a client name</p>
    <div class="pillar-row">
        <span class="pillar-tag p1">P1 &middot; TF-IDF &middot; 40%</span>
        <span class="pillar-tag p2">P2 &middot; Crime &middot; 30%</span>
        <span class="pillar-tag p4">P4 &middot; GraphSAGE &middot; 20%</span>
        <span class="pillar-tag p5">P5 &middot; CLIP &middot; 10%</span>
    </div>
</div>
""", unsafe_allow_html=True)

data = load_pipeline()

if "messages" not in st.session_state:
    st.session_state.messages = []
if "pending_screen" not in st.session_state:
    st.session_state.pending_screen = None

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Screen a client or ask about the fugitive database..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    rag_context = None
    user_msg = None
    run_screening = False
    client_bio = None

    if st.session_state.pending_screen:
        pending_name = st.session_state.pending_screen
        if api_key:
            oai = OpenAI(api_key=api_key)
            parse_msgs = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"The user previously asked to screen \"{pending_name}\" and was asked for biometric data. They replied with:\\n\\n\"{prompt}\"\\n\\nExtract the biometric fields and respond with the BIOMETRICS_COLLECTED line. If they did not provide enough info, ask again politely."},
            ]
            parse_resp = oai.chat.completions.create(model=model, messages=parse_msgs)
            llm_text = parse_resp.choices[0].message.content
            client_bio = parse_biometrics(llm_text)

        if client_bio:
            client_bio['full_name'] = pending_name
            rag_context, tier, score = screen(pending_name, data, client_bio=client_bio)
            bio_summary = ", ".join(f"{k}={v}" for k, v in client_bio.items() if k != 'full_name')
            user_msg = f"I screened client \"{pending_name}\" with biometrics: {bio_summary}.\\n\\nPipeline results:\\n\\n{rag_context}\\n\\nExplain this screening result clearly. What does each pillar score mean? Is this person a risk?"
            run_screening = True
            st.session_state.pending_screen = None
        else:
            user_msg = f"The user tried to provide biometric data for screening \"{pending_name}\" but the data was incomplete or unclear. Their message: \"{prompt}\". Ask them again for the missing fields: gender (M/F), date of birth or age, height (m), weight (kg), hair colour, eye colour."

    else:
        screen_match = re.match(r'(?i)screen\s+(.+)', prompt.strip())
        if screen_match:
            client_name = screen_match.group(1).strip().strip('"\'')
            st.session_state.pending_screen = client_name
            user_msg = f"The user wants to screen a client named \"{client_name}\" but has only provided the name. Ask them for the biometric data needed for P3 validation: gender, date of birth (or age), height (m), weight (kg), hair colour, and eye colour. Explain briefly why biometric data is needed (to avoid discrimination based solely on name matching and to power the P3 biometric gate)."
        else:
            rag_context = db_context(prompt, data)
            user_msg = f"The user asked: \"{prompt}\"\\n\\nHere is the current database context:\\n\\n{rag_context}\\n\\nAnswer their question using the data above."

    if not api_key:
        with st.chat_message("assistant"):
            st.warning("Enter your OpenAI API key in the sidebar for AI-powered explanations.")
            if rag_context:
                st.code(rag_context, language=None)
            fallback = f"**Raw pipeline output** (enter API key for AI explanation):\\n```\\n{(rag_context or 'Please provide biometric data.')[:2000]}\\n```"
            st.session_state.messages.append({"role": "assistant", "content": fallback})
    else:
        oai_client = OpenAI(api_key=api_key)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        with st.chat_message("assistant"):
            stream = oai_client.chat.completions.create(model=model, messages=messages, stream=True)
            response = st.write_stream(stream)

        if run_screening and rag_context:
            render_score_card(st.session_state.pending_screen or client_name if 'client_name' in dir() else 'Client', tier, score, rag_context)

        bio_from_response = parse_biometrics(response)
        if bio_from_response and st.session_state.pending_screen:
            pending_name = st.session_state.pending_screen
            bio_from_response['full_name'] = pending_name
            rag_context, tier, score = screen(pending_name, data, client_bio=bio_from_response)
            bio_summary = ", ".join(f"{k}={v}" for k, v in bio_from_response.items() if k != 'full_name')
            follow_msg = f"I screened client \"{pending_name}\" with biometrics: {bio_summary}.\\n\\nPipeline results:\\n\\n{rag_context}\\n\\nExplain this screening result clearly."
            st.session_state.pending_screen = None
            msgs2 = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": follow_msg}]
            with st.chat_message("assistant"):
                stream2 = oai_client.chat.completions.create(model=model, messages=msgs2, stream=True)
                response2 = st.write_stream(stream2)
                render_score_card(pending_name, tier, score, rag_context)
            st.session_state.messages.append({"role": "assistant", "content": response2})

        st.session_state.messages.append({"role": "assistant", "content": response})
'''

streamlit_path = os.path.join(os.getcwd(), 'streamlit_app.py')
with open(streamlit_path, 'w') as f:
    f.write(STREAMLIT_CODE.strip() + '\n')
print(f'✅ Streamlit app written to: {streamlit_path}')
print(f'   Run with: streamlit run streamlit_app.py')
print(f'   Requires: pip install streamlit openai')


✅ Streamlit app written to: /Volumes/My Mini Drive/CS 610/CS610_Interpol/Integration/streamlit_app.py
   Run with: streamlit run streamlit_app.py
   Requires: pip install streamlit openai
